## 1. import

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool

## 2. config

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }

## 3. column groups

In [3]:
# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE

DIGIT_SOURCE_COLS = [
    "Soil_pH",
    "Soil_Moisture",
    "Temperature_C",
    "Rainfall_mm",
    "Previous_Irrigation_mm",
]

## 4. utility

In [4]:
# =========================
# Utilities
# =========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def cast_dtypes(train_df, test_df):
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    return train_df, test_df

## 5. load data

In [5]:
seed_everything(CFG.SEED)

train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)

train_df, test_df = cast_dtypes(train_df, test_df)

print(train_df.shape, test_df.shape)

(630000, 21) (270000, 20)


## 6. label encode

In [6]:
le = LabelEncoder()
y = le.fit_transform(train_df[CFG.TARGET_COL])
class_names = list(le.classes_)
n_classes = len(class_names)

print("classes:", class_names)

classes: ['High', 'Low', 'Medium']


## 7. DIGIT feature 関数

In [7]:
def add_digit_features(df: pd.DataFrame, source_cols):
    df = df.copy()
    created_cols = []

    for col in source_cols:
        x = pd.to_numeric(df[col], errors="coerce").astype(float)

        integer_part = np.floor(x)
        frac_part = x - integer_part

        digit1 = np.floor(frac_part * 10 + 1e-8)
        digit2 = np.floor(frac_part * 100 + 1e-8) % 10

        round1 = np.round(x, 1)
        diff_round1 = np.abs(x - round1)

        integer_like = (np.abs(x - np.round(x, 0)) < 1e-8).astype(float)

        new_map = {
            f"digit__{col}__intpart": integer_part,
            f"digit__{col}__frac": frac_part,
            f"digit__{col}__d1": digit1,
            f"digit__{col}__d2": digit2,
            f"digit__{col}__diff_round1": diff_round1,
            f"digit__{col}__is_intlike": integer_like,
        }

        for new_col, values in new_map.items():
            df[new_col] = values
            created_cols.append(new_col)

    return df, created_cols

## 8. DIGIT feature 作成

In [8]:
train_df, digit_cols = add_digit_features(train_df, DIGIT_SOURCE_COLS)
test_df, _ = add_digit_features(test_df, DIGIT_SOURCE_COLS)

FEATURE_COLS_DIGIT = FEATURE_COLS_BASE + digit_cols

print("DIGIT_SOURCE_COLS:", DIGIT_SOURCE_COLS)
print("n digit cols:", len(digit_cols))
print("digit cols sample:", digit_cols[:10])

DIGIT_SOURCE_COLS: ['Soil_pH', 'Soil_Moisture', 'Temperature_C', 'Rainfall_mm', 'Previous_Irrigation_mm']
n digit cols: 30
digit cols sample: ['digit__Soil_pH__intpart', 'digit__Soil_pH__frac', 'digit__Soil_pH__d1', 'digit__Soil_pH__d2', 'digit__Soil_pH__diff_round1', 'digit__Soil_pH__is_intlike', 'digit__Soil_Moisture__intpart', 'digit__Soil_Moisture__frac', 'digit__Soil_Moisture__d1', 'digit__Soil_Moisture__d2']


## 9. CV

In [9]:
X = train_df[FEATURE_COLS_DIGIT].copy()
X_test = test_df[FEATURE_COLS_DIGIT].copy()

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 60)
    print(f"fold {fold} / {CFG.N_SPLITS}")
    print("=" * 60)

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_test,
        cat_features=CAT_ALL_BASE,
    )

    model = CatBoostClassifier(**CFG.MODEL_PARAMS)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba += te_proba / CFG.N_SPLITS

    va_pred = np.argmax(va_proba, axis=1)
    fold_bacc = balanced_accuracy_score(y_va, va_pred)
    fold_scores.append(fold_bacc)

    print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

fold 1 / 5
0:	learn: 1.0373202	test: 1.0372788	best: 1.0372788 (0)	total: 135ms	remaining: 6m 44s
200:	learn: 0.0625213	test: 0.0627919	best: 0.0627919 (200)	total: 3.17s	remaining: 44.1s
400:	learn: 0.0612168	test: 0.0618461	best: 0.0618461 (399)	total: 5.82s	remaining: 37.7s
600:	learn: 0.0595306	test: 0.0607860	best: 0.0607860 (600)	total: 8.56s	remaining: 34.2s
800:	learn: 0.0578051	test: 0.0599876	best: 0.0599876 (800)	total: 11.3s	remaining: 31.1s
1000:	learn: 0.0563732	test: 0.0594176	best: 0.0594176 (1000)	total: 14.1s	remaining: 28.1s
1200:	learn: 0.0553637	test: 0.0590411	best: 0.0590406 (1198)	total: 16.8s	remaining: 25.2s
1400:	learn: 0.0544372	test: 0.0587745	best: 0.0587745 (1400)	total: 19.4s	remaining: 22.2s
1600:	learn: 0.0536434	test: 0.0585241	best: 0.0585241 (1600)	total: 22.2s	remaining: 19.4s
1800:	learn: 0.0527179	test: 0.0582776	best: 0.0582749 (1798)	total: 25s	remaining: 16.6s
2000:	learn: 0.0518079	test: 0.0580722	best: 0.0580722 (2000)	total: 27.8s	remaining

## 10. summary

In [10]:
print("=" * 60)
print("fold scores:", [round(s, 6) for s in fold_scores])
print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

oof_pred = np.argmax(oof_proba, axis=1)
oof_bacc = balanced_accuracy_score(y, oof_pred)
print(f"OOF balanced_accuracy = {oof_bacc:.6f}")

print("\nPer-class recall on OOF:")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y == cls_idx)
    recall = (oof_pred[mask] == y[mask]).mean()
    print(f"{cls_name}: {recall:.6f}")

fold scores: [np.float64(0.95882), np.float64(0.96232), np.float64(0.961542), np.float64(0.959969), np.float64(0.960667)]
cv mean balanced_accuracy = 0.960664
cv std  balanced_accuracy = 0.001216
OOF balanced_accuracy = 0.960664

Per-class recall on OOF:
High: 0.912609
Low: 0.994985
Medium: 0.974397


## 11. save

In [11]:
test_pred = np.argmax(test_proba, axis=1)
test_label = le.inverse_transform(test_pred)

sub_df = pd.DataFrame({
    CFG.ID_COL: test_df[CFG.ID_COL],
    CFG.TARGET_COL: test_label
})

np.save("oof_catboost_digit_min_proba.npy", oof_proba)
np.save("pred_catboost_digit_min_proba.npy", test_proba)
sub_df.to_csv("submission_catboost_digit_min.csv", index=False)

print("\nSaved:")
print("- oof_catboost_digit_min_proba.npy")
print("- pred_catboost_digit_min_proba.npy")
print("- submission_catboost_digit_min.csv")


Saved:
- oof_catboost_digit_min_proba.npy
- pred_catboost_digit_min_proba.npy
- submission_catboost_digit_min.csv
